IMPORT LIBRARIES

In [2]:
import os, random, time, gc, torch, io, json, requests
import numpy as np
from PIL import Image
import easyocr
import pymupdf  
import kagglehub
import chromadb
from sentence_transformers import SentenceTransformer
from sklearn.metrics import ndcg_score

/home/jupyter-user/gender-classification-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DOWNLOAD DATASETS

In [3]:
print("📥 Downloading Dataset 1 (Old: hadikp/resume-data-pdf)...")
OLD_DATASET_DIR = kagglehub.dataset_download("hadikp/resume-data-pdf")

print("📥 Downloading Dataset 2 (New: snehaanbhawal/resume-dataset)...")
NEW_DATASET_DIR = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

print("\n Both datasets are successfully cached locally!")

📥 Downloading Dataset 1 (Old: hadikp/resume-data-pdf)...
📥 Downloading Dataset 2 (New: snehaanbhawal/resume-dataset)...

 Both datasets are successfully cached locally!


CONFIGURATION

In [5]:
# API Server Config
SERVER_URL = "http://173.208.247.17:8477/v1/chat/completions" 
HEADERS = {"Content-Type": "application/json"}

# Dataset Categories
OLD_CATEGORIES = ['Accountant', 'Agriculture', 'Aviation', 'Banking', 'Finance', 'HR', 'IT']
NEW_CATEGORIES = ['ACCOUNTANT', 'AGRICULTURE', 'AVIATION', 'BANKING', 'FINANCE', 'HR', 'INFORMATION-TECHNOLOGY']

SYSTEM PROMPTS

In [6]:
rubrics = {
    'ACCOUNTING': """Scoring Rubric:
- 1-2: No accounting, bookkeeping, or tax context whatsoever.
- 3-4: General administrative/business background with minimal invoicing or office math.
- 5-7: Entry to Mid-level accounting duties (AP/AR, payroll, bookkeeping, Excel reconciliation, tax filing support).
- 8-10: Senior accountant/auditor, CPA/CMA, complex financial reporting, or team leadership.""",
    'AGRICULTURE': """Scoring Rubric:
- 1-2: Completely unrelated domain with no agricultural relevance.
- 3-4: General manual labor or supply chain without direct farming context.
- 5-7: Hands-on field experience (crop cultivation, livestock care, soil testing, harvesting, farm machinery operation).
- 8-10: Agronomy specialist, farm operations supervisor, agricultural research, or precision agriculture expertise.""",
    'AVIATION': """Scoring Rubric:
- 1-2: No airline, airport, aircraft, or aviation industry context.
- 3-4: General hospitality/customer service without airline or airport background.
- 5-7: Airline customer service, flight attendant/cabin crew, baggage/ground operations, or basic ramp coordination.
- 8-10: Licensed pilot, FAA/CAA certifications, air traffic control, avionics technician, or aircraft maintenance engineer.""",
    'BANKING': """Scoring Rubric:
- 1-2: No banking, cash handling, or financial institution exposure.
- 3-4: General cashier or retail sales without banking compliance.
- 5-7: Bank teller, loan processing officer, customer relationship manager, credit analysis, or branch support.
- 8-10: Branch manager, senior underwriter, corporate banking relationship lead, or banking compliance officer.""",
    'FINANCE': """Scoring Rubric:
- 1-2: No finance, investment, or budget-related context.
- 3-4: General billing or basic data entry without financial reasoning.
- 5-7: Junior/Mid-level financial analysis, budgeting, forecasting, ledger management, or investment reporting.
- 8-10: Senior financial analyst, portfolio manager, CFA charterholder, or strategic finance director.""",
    'HR': """Scoring Rubric:
- 1-2: No human resources, recruitment, or people operations background.
- 3-4: General administrative support with minimal staff interaction.
- 5-7: HR generalist/assistant, recruiter, onboarding coordinator, employee benefits administrator, or training support.
- 8-10: Senior HR business partner, head of talent acquisition, organizational development, or labor law specialist.""",
    'IT': """Scoring Rubric:
- 1-2: Non-technical background with no IT or software exposure.
- 3-4: Basic computer literacy (MS Office, basic email) without technical administration.
- 5-7: IT support/helpdesk, hardware troubleshooting, basic scripting/coding, junior system administration, or web maintenance.
- 8-10: Senior software engineer, network architect, DevOps specialist, database administrator, or IT team lead."""
}

# Apply to New Dataset Prompts
new_role_prompts = {
    'ACCOUNTANT': f"You are an Accounting Hiring Manager. Evaluate this CV for any Accounting/Finance support role.\n{rubrics['ACCOUNTING']}",
    'AGRICULTURE': f"You are an Agricultural Operations Manager. Evaluate this CV for any Agriculture/Farming role.\n{rubrics['AGRICULTURE']}",
    'AVIATION': f"You are an Aviation Industry Recruiter. Evaluate this CV for any Aviation/Aerospace role.\n{rubrics['AVIATION']}",
    'BANKING': f"You are a Banking Recruitment Specialist. Evaluate this CV for any Retail or Commercial Banking position.\n{rubrics['BANKING']}",
    'FINANCE': f"You are a Corporate Finance Manager. Evaluate this CV for any Corporate Finance or Financial Advisory role.\n{rubrics['FINANCE']}",
    'HR': f"You are an HR Director. Evaluate this CV for any Human Resources or Talent Acquisition position.\n{rubrics['HR']}",
    'INFORMATION-TECHNOLOGY': f"You are an IT Support and Engineering Lead. Evaluate this CV for any Information Technology role.\n{rubrics['IT']}"
}

# Apply to Old Dataset Prompts
old_role_prompts = {
    'Accountant': new_role_prompts['ACCOUNTANT'],
    'Agriculture': new_role_prompts['AGRICULTURE'],
    'Aviation': new_role_prompts['AVIATION'],
    'Banking': new_role_prompts['BANKING'],
    'Finance': new_role_prompts['FINANCE'],
    'HR': new_role_prompts['HR'],
    'IT': new_role_prompts['INFORMATION-TECHNOLOGY']
}

OCR EXTRACTOR & SAVE MANIFEST

In [7]:
print(" Initializing EasyOCR Reader...")
reader = easyocr.Reader(['en'], gpu=True)

def run_ocr_extraction(dataset_base_dir, target_categories, dataset_label, manifest_save_path):
    source_dir = None
    for root, dirs, files in os.walk(dataset_base_dir):
        upper_dirs = [d.upper() for d in dirs]
        if sum(1 for cat in target_categories if cat.upper() in upper_dirs) >= 3:
            source_dir = root
            break
            
    if not source_dir:
        raise FileNotFoundError(f"Could not locate category directories in {dataset_base_dir}")
        
    print(f"\n [{dataset_label}] Scanning: {source_dir}")
    manifest = []
    total_start = time.time()
    cv_counter = 1
    
    for category in target_categories:
        actual_folder = next((d for d in os.listdir(source_dir) if d.upper() == category.upper()), None)
        if not actual_folder:
            continue
            
        cat_path = os.path.join(source_dir, actual_folder)
        pdf_files = [f for f in os.listdir(cat_path) if f.lower().endswith('.pdf')]
        random.shuffle(pdf_files)
        loaded = 0
        
        for file in pdf_files:
            if loaded >= 10:
                break
            
            cv_start = time.time()
            try:
                text_blocks = []
                with pymupdf.open(os.path.join(cat_path, file)) as pdf:
                    for page_num in range(len(pdf)):
                        pix = pdf.load_page(page_num).get_pixmap(dpi=90)
                        img = Image.open(io.BytesIO(pix.tobytes("png")))
                        results = reader.readtext(np.array(img), detail=0)
                        text_blocks.append(" ".join(results))
                        del pix, img, results
                
                raw_text = "\n".join(text_blocks)
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                
                cv_latency = time.time() - cv_start
                
                if len(raw_text.strip()) > 50:
                    manifest.append({
                        "category": category,
                        "filename": file,
                        "raw_text": raw_text
                    })
                    print(f"[{cv_counter}/70] ✅ OCR Success | {category} | {file} | Time: {cv_latency:.2f}s")
                    loaded += 1
                    cv_counter += 1
            except Exception as e:
                continue

    total_time = time.time() - total_start
    print(f"\n🎉 [{dataset_label}] Extraction Finished! Extracted {len(manifest)} CVs in {total_time / 60:.2f} min.")
    
    # Save manifest separately
    with open(manifest_save_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f"💾 Manifest saved to: {manifest_save_path}")
    
    return manifest

# 1. Run & Save Old Dataset Manifest
old_sampled_manifest = run_ocr_extraction(
    dataset_base_dir=OLD_DATASET_DIR,
    target_categories=OLD_CATEGORIES,
    dataset_label="Old Dataset (hadikp)",
    manifest_save_path="../data/old_sampled_manifest.json"
)

# 2. Run & Save New Dataset Manifest
new_sampled_manifest = run_ocr_extraction(
    dataset_base_dir=NEW_DATASET_DIR,
    target_categories=NEW_CATEGORIES,
    dataset_label="New Dataset (snehaanbhawal)",
    manifest_save_path="../data/new_sampled_manifest.json"
)

 Initializing EasyOCR Reader...

 [Old Dataset (hadikp)] Scanning: /home/jupyter-user/.cache/kagglehub/datasets/hadikp/resume-data-pdf/versions/1/Resumes PDF
[1/70] ✅ OCR Success | Accountant | bd25e35f4a574fd2.pdf | Time: 7.57s
[2/70] ✅ OCR Success | Accountant | 195.pdf | Time: 6.64s
[3/70] ✅ OCR Success | Accountant | 8.pdf | Time: 4.23s
[4/70] ✅ OCR Success | Accountant | 1315cc92aab2c895.pdf | Time: 3.43s
[5/70] ✅ OCR Success | Accountant | 43.pdf | Time: 5.84s
[6/70] ✅ OCR Success | Accountant | 177.pdf | Time: 6.03s
[7/70] ✅ OCR Success | Accountant | 0857decaec1c1f99.pdf | Time: 2.94s
[8/70] ✅ OCR Success | Accountant | 12.pdf | Time: 6.24s
[9/70] ✅ OCR Success | Accountant | 85.pdf | Time: 5.57s
[10/70] ✅ OCR Success | Accountant | 2.pdf | Time: 5.48s
[11/70] ✅ OCR Success | Agriculture | 49d8deebdf3819dc.pdf | Time: 2.35s
[12/70] ✅ OCR Success | Agriculture | 5d19cb8908789b70.pdf | Time: 3.87s
[13/70] ✅ OCR Success | Agriculture | 585c6f9d6b000b97.pdf | Time: 3.05s
[14/70] ✅ 

AUTOMATED SCORING ENGINE (LLM-AS-A-JUDGE)

In [8]:
def generate_ground_truth(manifest, role_prompts_dict, dataset_label, ground_truth_save_path):
    print(f" Starting LLM Scoring for: {dataset_label}")
    
    ground_truth = {}
    total_start = time.time()
    
    for idx, candidate in enumerate(manifest):
        doc_id = candidate['filename']
        category = candidate['category']
        ground_truth[doc_id] = {}
        
        print(f"\n[{idx + 1}/{len(manifest)}] Evaluating CV: {doc_id} (Category: {category})")
        cv_start = time.time()
        
        for role_name, prompt in role_prompts_dict.items():
            payload = {
                "model": "Qwen3.8-27B",
                "messages": [
                    {"role": "system", "content": prompt},
                    {
                        "role": "user",
                        "content": f"Evaluate the candidate CV against the role criteria and assign an integer score from 1 to 10 according to the rubric.\nOutput ONLY a JSON object: {{\"score\": <number>}}\n\nCV Text:\n{candidate['raw_text'][:5000]}"
                    }
                ],
                "temperature": 0.2,
                "chat_template_kwargs": {"enable_thinking": False}
            }
            
            try:
                response = requests.post(SERVER_URL, headers=HEADERS, json=payload, timeout=30).json()
                raw_reply = response['choices'][0]['message']['content']
                json_str = raw_reply[raw_reply.find("{"):raw_reply.rfind("}") + 1]
                score = int(json.loads(json_str).get("score", 1))
            except Exception:
                score = 1
                
            ground_truth[doc_id][role_name] = score
            print(f"   -> {role_name}: {score}/10")
            time.sleep(0.4)
            
        cv_latency = time.time() - cv_start
        print(f" Time taken for candidate {doc_id} across all 7 roles: {cv_latency:.2f} seconds")
        
    total_time = time.time() - total_start
    print(f"\n [{dataset_label}] Scoring Complete! Total time: {total_time / 60:.2f} minutes")
    
    # Save Ground Truth separately
    with open(ground_truth_save_path, "w", encoding="utf-8") as f:
        json.dump(ground_truth, f, ensure_ascii=False, indent=2)
    print(f" Ground Truth successfully saved to: {ground_truth_save_path}")
    
    return ground_truth

# 1. Score and Save Old Dataset Ground Truth
old_ground_truth = generate_ground_truth(
    manifest=old_sampled_manifest,
    role_prompts_dict=old_role_prompts,
    dataset_label="Old Dataset (hadikp)",
    ground_truth_save_path="../data/old_ocr_ground_truth.json"
)

# 2. Score and Save New Dataset Ground Truth
new_ground_truth = generate_ground_truth(
    manifest=new_sampled_manifest,
    role_prompts_dict=new_role_prompts,
    dataset_label="New Dataset (snehaanbhawal)",
    ground_truth_save_path="../data/new_ocr_ground_truth.json"
)

 Starting LLM Scoring for: Old Dataset (hadikp)

[1/70] Evaluating CV: bd25e35f4a574fd2.pdf (Category: Accountant)
   -> Accountant: 8/10
   -> Agriculture: 1/10
   -> Aviation: 1/10
   -> Banking: 5/10
   -> Finance: 6/10
   -> HR: 1/10
   -> IT: 3/10
 Time taken for candidate bd25e35f4a574fd2.pdf across all 7 roles: 25.05 seconds

[2/70] Evaluating CV: 195.pdf (Category: Accountant)
   -> Accountant: 8/10
   -> Agriculture: 1/10
   -> Aviation: 1/10
   -> Banking: 3/10
   -> Finance: 6/10
   -> HR: 3/10
   -> IT: 3/10
 Time taken for candidate 195.pdf across all 7 roles: 26.28 seconds

[3/70] Evaluating CV: 8.pdf (Category: Accountant)
   -> Accountant: 7/10
   -> Agriculture: 1/10
   -> Aviation: 1/10
   -> Banking: 5/10
   -> Finance: 6/10
   -> HR: 3/10
   -> IT: 3/10
 Time taken for candidate 8.pdf across all 7 roles: 24.76 seconds

[4/70] Evaluating CV: 1315cc92aab2c895.pdf (Category: Accountant)
   -> Accountant: 9/10
   -> Agriculture: 1/10
   -> Aviation: 1/10
   -> Banking: 

CURRENT VS PROPOSED HYBRID SYSTEM EVALUATION

In [21]:
import json
import os

print("🔄 Reloading saved data from disk...")

# Reload Old Dataset
if os.path.exists('../data/old_sampled_manifest.json') and os.path.exists('../data/old_ocr_ground_truth.json'):
    with open('../data/old_sampled_manifest.json', 'r') as f:
        old_sampled_manifest = json.load(f)
    with open('../data/old_ocr_ground_truth.json', 'r') as f:
        old_ground_truth = json.load(f)
    print("✅ Old Dataset loaded successfully!")
else:
    print("⚠️ Old Dataset JSON files not found. Did Cell 3 and 4 finish saving them?")

# Reload New Dataset
if os.path.exists('../data/new_sampled_manifest.json') and os.path.exists('../data/new_ocr_ground_truth.json'):
    with open('../data/new_sampled_manifest.json', 'r') as f:
        new_sampled_manifest = json.load(f)
    with open('../data/new_ocr_ground_truth.json', 'r') as f:
        new_ground_truth = json.load(f)
    print("✅ New Dataset loaded successfully!")
else:
    print("⚠️ New Dataset JSON files not found. Did Cell 3 and 4 finish saving them?")

🔄 Reloading saved data from disk...
✅ Old Dataset loaded successfully!
✅ New Dataset loaded successfully!


In [28]:
# 1. RELOAD DATA 
print(" Reloading saved data from disk...")
try:
    with open('../data/old_sampled_manifest.json', 'r') as f: old_sampled_manifest = json.load(f)
    with open('../data/old_ocr_ground_truth.json', 'r') as f: old_ground_truth = json.load(f)
    
    with open('../data/new_sampled_manifest.json', 'r') as f: new_sampled_manifest = json.load(f)
    with open('../data/new_ocr_ground_truth.json', 'r') as f: new_ground_truth = json.load(f)
    print(" All Data loaded successfully!\n")
except Exception as e:
    print(f" Error loading JSON files: {e}. Pastikan Cell 3 dan 4 sudah selesai dijalankan.")

# 2. SETUP MODEL & DATABASE
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

print("⏳ Loading BAAI/bge-large-en-v1.5...")
device = "cuda" if torch.cuda.is_available() else "cpu"
bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

chroma_client = chromadb.Client()

eval_queries = [
    "Senior Accountant CPA certifications tax compliance auditing payroll",
    "Agriculture role crop management farming operations soil science",
    "Aviation Professional FAA certifications flight operations",
    "Banking role retail banking loan processing regulatory compliance",
    "Financial Analyst financial modeling forecasting ROI analysis",
    "HR Manager talent acquisition employee relations payroll",
    "IT Specialist network administration software development"
]

# 3. EVALUATION FUNCTION
def evaluate_pipeline(manifest, ground_truth, collection_name, target_categories, dataset_label):
    if not manifest or len(manifest) == 0:
        return
        
    total_docs = len(manifest)
    #The text is transformed into vector representations using the BAAI/bge-large-en-v1.5 model:
    print(f"\n Ingesting {total_docs} documents for '{dataset_label}'...")

    try: chroma_client.delete_collection(name=collection_name)
    except: pass
        
    collection = chroma_client.create_collection(name=collection_name)
    
    docs = [c['raw_text'] for c in manifest]
    ids = [f"{c['category']}_{c['filename']}" for c in manifest] # UNIQUE IDs
    
    # new ChromaDB IDs to old Ground Truth keys
    mapped_gt = {}
    for c in manifest:
        combined_id = f"{c['category']}_{c['filename']}"
        base_id = c['filename']
        mapped_gt[combined_id] = ground_truth.get(base_id, {})

    embeddings = bge_model.encode(docs, normalize_embeddings=True, show_progress_bar=False).tolist()
    
    collection.add(
        documents=docs, embeddings=embeddings,
        metadatas=[{"category": c['category']} for c in manifest], ids=ids
    )
    
    baseline_ndcgs, hybrid_ndcgs, baseline_precs, hybrid_precs = [], [], [], []
    k_baseline, k_stage1 = min(10, total_docs), min(30, total_docs)
    
    print(f"\n=======================================================")
    print(f" EVALUATION RESULTS: {dataset_label.upper()}")
    print(f"=======================================================")
    
    for role, query_text in zip(target_categories, eval_queries):
        all_doc_ids = ids 
        true_scores = [mapped_gt[doc_id].get(role, 1) for doc_id in all_doc_ids]
        #Encodes the recruiter's search query into the same embedding space
        query_embedding = bge_model.encode([query_text], normalize_embeddings=True).tolist()
        
        # 1. Current Semantic Search
        #Current system queries the Top 10 most semantically similar documents
        b_res = collection.query(query_embeddings=query_embedding, n_results=k_baseline)
        b_ids, b_dists = b_res['ids'][0], b_res['distances'][0]
        #Convert distance 'd' into a similarity score between 0 and 1
        b_scores = [1.0 / (1.0 + d) for d in b_dists]
        
        # 2. Proposed Hybrid System (Top-30 reranked)
        #Hybrid system queries the Top 30 candidate pool
        h_stage1 = collection.query(query_embeddings=query_embedding, n_results=k_stage1)
        hybrid_candidates = []
        for d_id, dist in zip(h_stage1['ids'][0], h_stage1['distances'][0]):
            sem_score = 1.0 / (1.0 + dist) # 1.Semantic Similarity Score
            llm_score = mapped_gt.get(d_id, {}).get(role, 1) / 10.0 # 2.LLM Qualitative Score normalized to 0-1
            hybrid_candidates.append({'id': d_id, 'score': (0.4 * sem_score) + (0.6 * llm_score)}) # 3. Combination (40% Semantic Similarity + 60% LLM Judge)
            
        hybrid_candidates = sorted(hybrid_candidates, key=lambda x: x['score'], reverse=True)[:k_baseline]
        h_ids = [c['id'] for c in hybrid_candidates]
        h_scores = [c['score'] for c in hybrid_candidates]
        
        # Metrics
        b_prec = sum(1 for d in b_ids if mapped_gt.get(d, {}).get(role, 1) >= 7) / float(k_baseline)
        h_prec = sum(1 for d in h_ids if mapped_gt.get(d, {}).get(role, 1) >= 7) / float(k_baseline)
        
        b_preds = [b_scores[b_ids.index(d)] if d in b_ids else 0 for d in all_doc_ids]
        h_preds = [h_scores[h_ids.index(d)] if d in h_ids else 0 for d in all_doc_ids]
        
        b_ndcg = ndcg_score([true_scores], [b_preds], k=k_baseline)
        h_ndcg = ndcg_score([true_scores], [h_preds], k=k_baseline)
        
        baseline_ndcgs.append(b_ndcg); hybrid_ndcgs.append(h_ndcg)
        baseline_precs.append(b_prec); hybrid_precs.append(h_prec)
        
        print(f"Role: {role:<24} | Baseline (NDCG: {b_ndcg:.3f}, P@{k_baseline}: {b_prec:.2f}) | Hybrid (NDCG: {h_ndcg:.3f}, P@{k_baseline}: {h_prec:.2f})")
        
    print("-------------------------------------------------------")
    print(f"AVERAGE CURRENT -> NDCG@{k_baseline}: {np.mean(baseline_ndcgs):.3f} | Precision@{k_baseline}: {np.mean(baseline_precs):.3f}")
    print(f"AVERAGE HYBRID   -> NDCG@{k_baseline}: {np.mean(hybrid_ndcgs):.3f} | Precision@{k_baseline}: {np.mean(hybrid_precs):.3f}")
    print("=======================================================\n")

# 4. EXECUTE THE FUNCTION 
if 'old_sampled_manifest' in globals():
    evaluate_pipeline(old_sampled_manifest, old_ground_truth, "old_collection", OLD_CATEGORIES, "Old Dataset (hadikp)")

if 'new_sampled_manifest' in globals():
    evaluate_pipeline(new_sampled_manifest, new_ground_truth, "new_collection", NEW_CATEGORIES, "New Dataset (snehaanbhawal)")

 Reloading saved data from disk...
 All Data loaded successfully!

⏳ Loading BAAI/bge-large-en-v1.5...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3022.87it/s]



 Ingesting 70 documents for 'Old Dataset (hadikp)'...

 EVALUATION RESULTS: OLD DATASET (HADIKP)
Role: Accountant               | Baseline (NDCG: 0.921, P@10: 0.90) | Hybrid (NDCG: 1.000, P@10: 1.00)
Role: Agriculture              | Baseline (NDCG: 0.908, P@10: 0.60) | Hybrid (NDCG: 1.000, P@10: 0.60)
Role: Aviation                 | Baseline (NDCG: 0.859, P@10: 0.80) | Hybrid (NDCG: 1.000, P@10: 1.00)
Role: Banking                  | Baseline (NDCG: 0.902, P@10: 0.70) | Hybrid (NDCG: 1.000, P@10: 1.00)
Role: Finance                  | Baseline (NDCG: 0.905, P@10: 0.50) | Hybrid (NDCG: 0.952, P@10: 0.60)
Role: HR                       | Baseline (NDCG: 0.928, P@10: 0.70) | Hybrid (NDCG: 1.000, P@10: 0.70)
Role: IT                       | Baseline (NDCG: 0.887, P@10: 0.50) | Hybrid (NDCG: 0.981, P@10: 0.50)
-------------------------------------------------------
AVERAGE CURRENT -> NDCG@10: 0.902 | Precision@10: 0.671
AVERAGE HYBRID   -> NDCG@10: 0.990 | Precision@10: 0.771


 Ingesting

In [31]:
import json
import numpy as np
from scipy.stats import spearmanr, kendalltau

print(" Loading Ground Truth matrices...")
try:
    # Text-based (CSV) Ground Truth from yesterday
    with open('../data/pseudo_qrels_ground_truth.json', 'r') as f:
        text_gt = json.load(f)
        
    # OCR-based Ground Truth from today
    with open('../data/new_ocr_ground_truth.json', 'r') as f:
        ocr_gt = json.load(f)
    print(" Files loaded successfully!")
except Exception as e:
    print(f" Error loading files: {e}. Check your file paths.")

# The 7 target roles evaluated in the New Dataset
roles = ['ACCOUNTANT', 'AGRICULTURE', 'AVIATION', 'BANKING', 'FINANCE', 'HR', 'INFORMATION-TECHNOLOGY']

text_scores_flat = []
ocr_scores_flat = []
role_correlations = {}

for role in roles:
    role_text_scores = []
    role_ocr_scores = []
    
    for ocr_id, ocr_scores in ocr_gt.items():
        # Match IDs: OCR IDs have '.pdf' (e.g., '123.pdf'), CSV IDs might just be '123'
        text_id = ocr_id if ocr_id in text_gt else ocr_id.replace('.pdf', '')
        
        if text_id in text_gt:
            # Extract the 1-10 scores
            t_score = text_gt[text_id].get(role, 1)
            o_score = ocr_scores.get(role, 1)
            
            role_text_scores.append(t_score)
            role_ocr_scores.append(o_score)
            
            text_scores_flat.append(t_score)
            ocr_scores_flat.append(o_score)
            
    # Calculate Correlation if we have enough matching pairs
    if len(role_text_scores) > 1:
        spearman_corr, _ = spearmanr(role_text_scores, role_ocr_scores)
        kendall_corr, _ = kendalltau(role_text_scores, role_ocr_scores)
        
        role_correlations[role] = {
            'spearman': spearman_corr,
            'kendall': kendall_corr,
            'count': len(role_text_scores)
        }

print("\n=======================================================")
print(" LLM SCORING CORRELATION: TEXT vs. OCR")
print("=======================================================")
for role, metrics in role_correlations.items():
    print(f"Role: {role:<22} | Spearman: {metrics['spearman']:.3f} | Kendall Tau: {metrics['kendall']:.3f}")
    
if len(text_scores_flat) > 1:
    total_spearman, _ = spearmanr(text_scores_flat, ocr_scores_flat)
    total_kendall, _ = kendalltau(text_scores_flat, ocr_scores_flat)
    print("-------------------------------------------------------")
    print(f" OVERALL CORRELATION (Matched Pairs: {len(text_scores_flat)})")
    print(f"Spearman's Rho : {total_spearman:.3f}")
    print(f"Kendall's Tau  : {total_kendall:.3f}")
    print("=======================================================\n")

 Loading Ground Truth matrices...
 Files loaded successfully!

 LLM SCORING CORRELATION: TEXT vs. OCR
Role: ACCOUNTANT             | Spearman: 0.957 | Kendall Tau: 0.920
Role: AGRICULTURE            | Spearman: 1.000 | Kendall Tau: 1.000
Role: AVIATION               | Spearman: 0.993 | Kendall Tau: 0.977
Role: BANKING                | Spearman: 0.956 | Kendall Tau: 0.898
Role: FINANCE                | Spearman: 0.969 | Kendall Tau: 0.941
Role: HR                     | Spearman: 0.781 | Kendall Tau: 0.715
Role: INFORMATION-TECHNOLOGY | Spearman: 0.907 | Kendall Tau: 0.845
-------------------------------------------------------
 OVERALL CORRELATION (Matched Pairs: 56)
Spearman's Rho : 0.936
Kendall's Tau  : 0.873

